In [ ]:
%pip -q install feedparser pandas tqdm
import re
import time
import random
import urllib.parse
import unicodedata
import requests
import feedparser
import pandas as pd
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo
from email.utils import parsedate_to_datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from google.colab import files
START_YEAR = 2017
END_YEAR = 2026
OUTPUT_FILE = 'AIEDnews_2017_2026_ROUND1_CANDIDATES.csv'
RSS_SPLIT_THRESHOLD = 95
MAX_SPLIT_DEPTH = 1
MIN_DATE_RANGE_DAYS = 90
MAX_REQUESTS_PER_TASK = 4
FETCH_WORKERS = 4
REQUEST_TIMEOUT = 12
REQUEST_RETRIES = 3
OUTLET_MAP = {'thanhnien.vn': 'Báo Thanh Niên', 'nhandan.vn': 'Báo Nhân Dân', 'laodong.vn': 'Báo Lao Động', 'vov.vn': 'VOV', 'giaoducthoidai.vn': 'Giáo dục & Thời đại', 'vnexpress.net': 'VnExpress', 'dantri.com.vn': 'Báo Dân trí', 'vietnamnet.vn': 'Báo VietNamNet', 'tuoitre.vn': 'Tuổi Trẻ', '24h.com.vn': '24h.com.vn', 'kenh14.vn': 'Kenh14.vn', 'znews.vn': 'ZNews', 'giaoduc.net.vn': 'Giáo dục Việt Nam', 'vtv.vn': 'VTV.vn', 'baochinhphu.vn': 'Báo Chính phủ', 'vtcnews.vn': 'VTC News', 'qdnd.vn': 'Báo Quân đội nhân dân', 'vietnamplus.vn': 'VietnamPlus', 'cafef.vn': 'CafeF', 'tiasang.com.vn': 'Tia Sáng', 'khoahocphattrien.vn': 'Khoa học & Phát triển'}
today = datetime.now(ZoneInfo('Asia/Ho_Chi_Minh')).date()
START_DATE = date(START_YEAR, 1, 1)
END_DATE = min(today + timedelta(days=1), date(END_YEAR + 1, 1, 1))
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36'}

def make_or_group(terms):
    return '(' + ' OR '.join((f'"{term}"' for term in terms)) + ')'

def build_queries():
    EDU_ANCHOR = make_or_group(['giáo dục', 'đại học', 'học sinh', 'sinh viên', 'trường học', 'giáo viên'])
    AI_2017_2021 = [['trí tuệ nhân tạo', 'công nghệ AI', 'ứng dụng AI'], ['học máy', 'học sâu', 'xử lý ngôn ngữ tự nhiên']]
    AI_2022_2023 = [['trí tuệ nhân tạo', 'AI tạo sinh', 'AI sinh tạo'], ['ChatGPT', 'OpenAI', 'GPT-3', 'GPT-4'], ['Google Bard', 'Bing Chat', 'Copilot', 'mô hình ngôn ngữ lớn']]
    AI_2024_2026 = [['trí tuệ nhân tạo', 'AI tạo sinh', 'trợ lý AI'], ['ChatGPT', 'OpenAI', 'GPT-4o', 'GPT-5', 'Gemini', 'Google Gemini'], ['Claude', 'DeepSeek', 'Perplexity AI', 'NotebookLM'], ['mô hình ngôn ngữ lớn', 'LLM', 'tác nhân AI', 'AI agent', 'chatbot AI']]
    SHORT_AI = ['AI', 'GPT', 'LLM']
    SPECIAL_CONTEXTS = [(make_or_group(['ChatGPT', 'AI', 'trí tuệ nhân tạo']), make_or_group(['đạo văn', 'gian lận thi cử', 'bài luận', 'luận văn', 'đồ án'])), (make_or_group(['ChatGPT', 'AI', 'trí tuệ nhân tạo']), make_or_group(['kỳ thi', 'đề thi', 'chấm bài', 'bài tập', 'thi cử']))]
    periods = [(date(2017, 1, 1), date(2022, 1, 1), AI_2017_2021), (date(2022, 1, 1), date(2024, 1, 1), AI_2022_2023), (date(2024, 1, 1), END_DATE, AI_2024_2026)]
    tasks = []
    for domain in OUTLET_MAP:
        for start_date, end_date, ai_groups in periods:
            if start_date >= END_DATE:
                continue
            end_date = min(end_date, END_DATE)
            for group in ai_groups:
                tasks.append({'domain': domain, 'query': make_or_group(group), 'start_date': start_date, 'end_date': end_date})
            for term in SHORT_AI:
                tasks.append({'domain': domain, 'query': f'"{term}" {EDU_ANCHOR}', 'start_date': start_date, 'end_date': end_date})
        for ai_query, edu_query in SPECIAL_CONTEXTS:
            tasks.append({'domain': domain, 'query': f'{ai_query} {edu_query}', 'start_date': START_DATE, 'end_date': END_DATE})
    return tasks

def clean_title(title):
    if not title:
        return ''
    title = unicodedata.normalize('NFC', str(title))
    title = re.sub('\\s+', ' ', title).strip()
    if ' - ' in title:
        title = title.rsplit(' - ', 1)[0].strip()
    return title

def parse_rfc822_date(pub_str):
    if not pub_str or pub_str == 'NO DATE':
        return (None, None, None)
    try:
        dt = parsedate_to_datetime(str(pub_str))
        return (dt.day, dt.month, dt.year)
    except Exception:
        return (None, None, None)

def join_unique(values):
    result = []
    for value in values:
        if pd.isna(value):
            continue
        for part in str(value).split(' || '):
            part = part.strip()
            if part and part not in result:
                result.append(part)
    return ' || '.join(result)

def request_rss(session, domain, query, start_date, end_date, budget):
    after_date = start_date - timedelta(days=1)
    full_query = f'site:{domain} {query} after:{after_date.isoformat()} before:{end_date.isoformat()}'
    params = {'q': full_query, 'hl': 'vi-VN', 'gl': 'VN', 'ceid': 'VN:vi'}
    url = 'https://news.google.com/rss/search?' + urllib.parse.urlencode(params)
    last_error = None
    for attempt in range(REQUEST_RETRIES):
        if budget['used'] >= MAX_REQUESTS_PER_TASK:
            raise RuntimeError('Request budget exhausted')
        budget['used'] += 1
        time.sleep(random.uniform(0.15, 0.4))
        try:
            response = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            if response.status_code == 429:
                if attempt < REQUEST_RETRIES - 1:
                    time.sleep(3 * (attempt + 1) + random.uniform(0.5, 1.5))
                continue
            if response.status_code in {500, 502, 503, 504}:
                if attempt < REQUEST_RETRIES - 1:
                    time.sleep(1.5 * (attempt + 1))
                continue
            if 400 <= response.status_code < 500:
                raise RuntimeError(f'HTTP {response.status_code}')
            response.raise_for_status()
            feed = feedparser.parse(response.content)
            return feed.entries
        except requests.RequestException as e:
            last_error = e
            if attempt < REQUEST_RETRIES - 1:
                time.sleep(1.5 * (attempt + 1) + random.uniform(0.2, 0.5))
    raise RuntimeError(last_error or 'RSS request failed')

def fetch_window(session, domain, query, start_date, end_date, budget, depth=0):
    entries = request_rss(session, domain, query, start_date, end_date, budget)
    days_diff = (end_date - start_date).days
    remaining_budget = MAX_REQUESTS_PER_TASK - budget['used']
    should_split = len(entries) >= RSS_SPLIT_THRESHOLD and depth < MAX_SPLIT_DEPTH and (days_diff > MIN_DATE_RANGE_DAYS) and (remaining_budget >= 2)
    if should_split:
        mid = start_date + timedelta(days=days_diff // 2)
        left = fetch_window(session, domain, query, start_date, mid, budget, depth + 1)
        right = []
        if budget['used'] < MAX_REQUESTS_PER_TASK:
            right = fetch_window(session, domain, query, mid, end_date, budget, depth + 1)
        return left + right
    results = []
    for entry in entries:
        title = clean_title(entry.get('title', ''))
        link = entry.get('link', '')
        if not title or not link:
            continue
        results.append({'title': title, 'google_news_url': link, 'pub_date_raw': entry.get('published', 'NO DATE'), 'outlet_domain': domain, 'discovery_query': query})
    return results

def fetch_task(task):
    budget = {'used': 0}
    with requests.Session() as session:
        return fetch_window(session=session, domain=task['domain'], query=task['query'], start_date=task['start_date'], end_date=task['end_date'], budget=budget, depth=0)

def run_round1():
    tasks = build_queries()
    print(f'Top-level tasks: {len(tasks):,}')
    print(f'Maximum HTTP requests/task: {MAX_REQUESTS_PER_TASK}')
    all_hits = []
    failed_tasks = 0
    with ThreadPoolExecutor(max_workers=FETCH_WORKERS) as executor:
        futures = [executor.submit(fetch_task, task) for task in tasks]
        for future in tqdm(as_completed(futures), total=len(futures), desc='RSS Harvesting'):
            try:
                result = future.result()
                if result:
                    all_hits.extend(result)
            except Exception:
                failed_tasks += 1
    df = pd.DataFrame(all_hits)
    if df.empty:
        print('Không thu thập được dữ liệu.')
        return
    records = []
    for (domain, google_news_url), group in df.groupby(['outlet_domain', 'google_news_url'], sort=False):
        first = group.iloc[0]
        day, month, year = parse_rfc822_date(first['pub_date_raw'])
        records.append({'title': first['title'], 'day': day, 'month': month, 'year': year, 'outlet_domain': domain, 'outlet_name': OUTLET_MAP.get(domain, domain), 'google_news_url': google_news_url, 'discovery_query': join_unique(group['discovery_query'])})
    final_df = pd.DataFrame(records)
    final_df = final_df[final_df['year'].isna() | final_df['year'].between(START_YEAR, END_YEAR)].copy()
    final_df = final_df.sort_values(['year', 'month', 'day', 'outlet_domain'], na_position='last').reset_index(drop=True)
    final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print()
    print(f'Hoàn tất Round 1: {len(final_df):,} candidates')
    print(f'Tasks thất bại: {failed_tasks:,}')
    print()
    print(final_df['year'].value_counts(dropna=False).sort_index())
    files.download(OUTPUT_FILE)
run_round1()
import re
import unicodedata
import pandas as pd
from google.colab import files


In [ ]:
INPUT_FILE = 'AIEDnews_2017_2026_ROUND1_CANDIDATES.csv'
OUTPUT_FILE = 'AIEDnews_2017_2026_STRICT_ANALYTICAL_CORPUS.csv'
df = pd.read_csv(INPUT_FILE)

def normalize_text(text):
    text = '' if pd.isna(text) else str(text)
    text = unicodedata.normalize('NFC', text)
    return re.sub('\\s+', ' ', text).strip()
PATTERN_AI_TOKEN = re.compile('(?<!\\w)(?:AI|A\\.I\\.)(?!\\w)')
PATTERN_AI = re.compile('\n    trí\\s*tuệ\\s*nhân\\s*tạo\n\n    |công\\s*nghệ\\s+ai\n    |ứng\\s*dụng\\s+ai\n    |công\\s*cụ\\s+ai\n    |mô\\s*hình\\s+ai\n    |hệ\\s*thống\\s+ai\n    |trợ\\s*lý\\s+ai\n\n    |ai\\s+tạo\\s+sinh\n    |ai\\s+sinh\\s+tạo\n    |trí\\s*tuệ\\s*nhân\\s*tạo\\s+tạo\\s+sinh\n\n    |học\\s*máy\n    |học\\s*sâu\n\n    |xử\\s*lý\\s+ngôn\\s*ngữ\\s+tự\\s*nhiên\n    |mô\\s*hình\\s+ngôn\\s*ngữ\\s+lớn\n    |mô\\s*hình\\s+ngôn\\s*ngữ\n\n    |mạng\\s*thần\\s*kinh\n    |mạng\\s*nơ[\\-\\s]*ron\n    |mạng\\s*neuron\n\n    |thị\\s*giác\\s*máy\\s*tính\n\n    |chat\\s*gpt\n    |openai\n\n    |\\bgpt\n        (?:[-\\s]?\n            (?:\n                3(?:\\.5)?\n                |4o?\n                |5\n            )\n        )?\n    \\b\n\n    |\\bgemini\\b\n    |google\\s+gemini\n\n    |\\bbard\\b\n    |google\\s+bard\n\n    |\\bcopilot\\b\n    |microsoft\\s+copilot\n\n    |bing\\s+chat\n\n    |\\bclaude\\b\n    |\\banthropic\\b\n\n    |\\bdeepseek\\b\n    |\\bgrok\\b\n\n    |meta\\s+ai\n    |notebooklm\n    |perplexity\\s+ai\n    |mistral\\s+ai\n    |\\bqwen\\b\n\n    |\\bllms?\\b\n\n    |tác\\s*nhân\\s+ai\n    |tác\\s*tử\\s+ai\n\n    |chatbot\\s+ai\n    ', re.IGNORECASE | re.VERBOSE)
PATTERN_EDU = re.compile('\n    giáo\\s*dục\n\n    |đại\\s*học\n    |cao\\s*đẳng\n    |học\\s*viện\n\n    |trường\\s*học\n    |nhà\\s*trường\n\n    |mầm\\s*non\n    |tiểu\\s*học\n    |trung\\s*học\n    |trung\\s*học\\s*cơ\\s*sở\n    |trung\\s*học\\s*phổ\\s*thông\n    |\\bthcs\\b\n    |\\bthpt\\b\n    |phổ\\s*thông\n\n    |học\\s*sinh\n    |sinh\\s*viên\n    |nghiên\\s*cứu\\s*sinh\n    |người\\s*học\n\n    |giáo\\s*viên\n    |giảng\\s*viên\n    |nhà\\s*giáo\n\n    |giảng\\s*dạy\n    |dạy\\s*học\n    |học\\s*tập\n\n    |lớp\\s*học\n    |giảng\\s*đường\n    |học\\s*đường\n\n    |khóa\\s*học\n    |môn\\s*học\n\n    |chương\\s*trình\\s*học\n    |chương\\s*trình\\s*đào\\s*tạo\n    |chương\\s*trình\\s*giáo\\s*dục\n    |cơ\\s*sở\\s*đào\\s*tạo\n\n    |đào\\s*tạo\\s+\n        (?:\n            học\\s*sinh\n            |sinh\\s*viên\n            |giáo\\s*viên\n            |giảng\\s*viên\n            |nhà\\s*giáo\n            |nhân\\s*lực\n            |nguồn\\s*nhân\\s*lực\n        )\n\n    |giáo\\s*trình\n    |sách\\s*giáo\\s*khoa\n    |giáo\\s*án\n    |bài\\s*giảng\n\n    |tuyển\\s*sinh\n    |học\\s*bổng\n\n    |bài\\s*kiểm\\s*tra\n    |thi\\s*cử\n    |kỳ\\s*thi\n    |bài\\s*thi\n    |đề\\s*thi\n    |chấm\\s*bài\n\n    |bài\\s*tập\n    |bài\\s*luận\n\n    |luận\\s*văn\n    |luận\\s*án\n    |khóa\\s*luận\n    |đồ\\s*án\n\n    |đạo\\s*văn\n    |gian\\s*lận\\s*học\\s*thuật\n    |gian\\s*lận\\s*thi\\s*cử\n    ', re.IGNORECASE | re.VERBOSE)

def get_matches(pattern, text):
    return list(dict.fromkeys((re.sub('\\s+', ' ', m.group(0)).strip() for m in pattern.finditer(text))))

def filter_title(title):
    title = normalize_text(title)
    ai_matches = get_matches(PATTERN_AI, title)
    ai_token = get_matches(PATTERN_AI_TOKEN, title)
    edu_matches = get_matches(PATTERN_EDU, title)
    ai_matches = list(dict.fromkeys(ai_token + ai_matches))
    return pd.Series({'ai_match': ' | '.join(ai_matches), 'edu_match': ' | '.join(edu_matches), 'keep': bool(ai_matches) and bool(edu_matches)})
matches = df['title'].apply(filter_title)
df = pd.concat([df, matches], axis=1)
df = df[df['keep']].drop(columns='keep').reset_index(drop=True)
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Giữ lại: {len(df):,} bài')
print(df['year'].value_counts(dropna=False).sort_index())
files.download(OUTPUT_FILE)


In [ ]:
%pip -q install -U googlenewsdecoder pandas tqdm
import pandas as pd
import time
import random
import os
from googlenewsdecoder import gnewsdecoder
from tqdm.notebook import tqdm
from google.colab import files
INPUT_FILE = 'AIEDnews_2017_2026_STRICT_ANALYTICAL_CORPUS.csv'
OUTPUT_FILE = 'AIEDnews_2017_2026_STRICT_WITH_REAL_URL.csv'
BATCH_SIZE = 100
BATCH_PAUSE = 35
SLEEP_BETWEEN_REQ = (0.6, 1.2)

def decode_single_url(url, retries=2):
    if pd.isna(url) or not str(url).strip():
        return (None, 'empty_url')
    url = str(url).strip()
    if 'news.google.com' not in url:
        return (url, 'already_real_url')
    for attempt in range(retries):
        try:
            result = gnewsdecoder(url, interval=0.2)
            if isinstance(result, dict) and result.get('status') and result.get('decoded_url'):
                return (result['decoded_url'], 'success')
            elif isinstance(result, str) and result.startswith('http'):
                return (result, 'success')
        except Exception as e:
            if '429' in str(e):
                time.sleep(5)
        time.sleep(random.uniform(0.5, 1.0))
    return (None, 'failed_or_429')
df = pd.read_csv(INPUT_FILE)
decoded_map = {}
status_map = {}
urls = df['google_news_url'].dropna().astype(str).str.strip().drop_duplicates().tolist()
print(f'Tổng số URL cần decode: {len(urls):,} URLs duy nhất')
batches = [urls[i:i + BATCH_SIZE] for i in range(0, len(urls), BATCH_SIZE)]
print(f'Đã chia thành {len(batches)} lô (Mỗi lô {BATCH_SIZE} URLs)\n')
total_success = 0
total_failed = 0
for b_idx, batch in enumerate(batches):
    print(f'\n==================================================')
    print(f'--- ĐANG XỬ LÝ LÔ {b_idx + 1}/{len(batches)} ({len(batch)} URLs) ---')
    batch_success = 0
    batch_failed = 0
    for url in tqdm(batch, desc=f'Lô {b_idx + 1}'):
        real_url, status = decode_single_url(url)
        decoded_map[url] = real_url
        status_map[url] = status
        if status in ['success', 'already_real_url']:
            batch_success += 1
        else:
            batch_failed += 1
        time.sleep(random.uniform(*SLEEP_BETWEEN_REQ))
    total_success += batch_success
    total_failed += batch_failed
    print(f'📊 KẾT QUẢ LÔ {b_idx + 1}:')
    print(f'   - Thành công: {batch_success}/{len(batch)} ({batch_success / len(batch) * 100:.1f}%)')
    print(f'   - Thất bại  : {batch_failed}/{len(batch)}')
    print(f'📈 TỔNG LŨY KẾ : {total_success}/{total_success + total_failed} URL ({total_success / (total_success + total_failed) * 100:.1f}%)')
    if b_idx < len(batches) - 1:
        print(f'--> Tạm nghỉ {BATCH_PAUSE}s để tránh Google chặn...')
        time.sleep(BATCH_PAUSE)
original_url = df['google_news_url']
df['real_url'] = original_url.map(decoded_map)
df['decode_status'] = original_url.map(status_map)
df['decode_success'] = df['real_url'].notna()
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print('\n🎉 ĐÃ HOÀN THÀNH TOÀN BỘ!')
files.download(OUTPUT_FILE)


In [ ]:
%pip -q install newspaper3k lxml_html_clean beautifulsoup4 cloudscraper selenium webdriver-manager
import re
import json
import time
import random
import shutil
import pandas as pd
import requests
import cloudscraper
from bs4 import BeautifulSoup
from newspaper import Article, Config
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from google.colab import files
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
INPUT_FILE = 'AIEDnews_2017_2026_STRICT_WITH_REAL_URL.csv'
OUTPUT_FILE = 'AIEDnews_2017_2026_FULLTEXT.csv'
MIN_TEXT_CHARS = 150
SIMPLE_PASSES = 2
NEWSPAPER_WORKERS = 6
REQUEST_WORKERS = 4
REQUEST_TIMEOUT = 12
df = pd.read_csv(INPUT_FILE)
if 'full_text' not in df.columns:
    df['full_text'] = ''
if 'char_count' not in df.columns:
    df['char_count'] = 0
if 'scrape_method' not in df.columns:
    df['scrape_method'] = ''
df['full_text'] = df['full_text'].fillna('')
df['char_count'] = pd.to_numeric(df['char_count'], errors='coerce').fillna(0)
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36', 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8', 'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.7'}

def is_missing(row):
    text = str(row.get('full_text', '') or '').strip()
    return not text or len(text) < MIN_TEXT_CHARS

def clean_text(text):
    if not text:
        return None
    text = str(text)
    lines = [re.sub('\\s+', ' ', x).strip() for x in text.splitlines()]
    lines = [x for x in lines if x]
    seen = set()
    unique = []
    for line in lines:
        key = line.casefold()
        if key not in seen:
            seen.add(key)
            unique.append(line)
    text = '\n\n'.join(unique).strip()
    if len(text) < MIN_TEXT_CHARS:
        return None
    return text

def save_progress():
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

def update_row(idx, text, method):
    text = clean_text(text)
    if not text:
        return False
    current = str(df.at[idx, 'full_text'] or '')
    if len(text) <= len(current):
        return False
    df.at[idx, 'full_text'] = text
    df.at[idx, 'char_count'] = len(text)
    df.at[idx, 'scrape_method'] = method
    return True

def missing_indices():
    return [idx for idx, row in df.iterrows() if is_missing(row) and pd.notna(row.get('real_url')) and str(row.get('real_url')).strip()]
config = Config()
config.browser_user_agent = HEADERS['User-Agent']
config.request_timeout = REQUEST_TIMEOUT

def scrape_newspaper(url):
    try:
        article = Article(url, config=config)
        article.download()
        article.parse()
        return clean_text(article.text)
    except Exception:
        return None

def run_newspaper_pass():
    targets = missing_indices()
    if not targets:
        return 0
    recovered = 0
    with ThreadPoolExecutor(max_workers=NEWSPAPER_WORKERS) as executor:
        future_map = {executor.submit(scrape_newspaper, df.at[idx, 'real_url']): idx for idx in targets}
        for future in tqdm(as_completed(future_map), total=len(future_map), desc='Newspaper3k'):
            idx = future_map[future]
            try:
                text = future.result()
                if update_row(idx, text, 'newspaper3k'):
                    recovered += 1
            except Exception:
                pass
    return recovered

def walk_json(obj):
    if isinstance(obj, dict):
        types = obj.get('@type', [])
        if isinstance(types, str):
            types = [types]
        if any((t in {'Article', 'NewsArticle', 'ReportageNewsArticle'} for t in types)):
            body = obj.get('articleBody')
            if body:
                yield body
        for value in obj.values():
            yield from walk_json(value)
    elif isinstance(obj, list):
        for item in obj:
            yield from walk_json(item)

def extract_json_ld(soup):
    for script in soup.find_all('script', type='application/ld+json'):
        raw = script.string or script.get_text()
        if not raw:
            continue
        try:
            data = json.loads(raw)
            for body in walk_json(data):
                text = clean_text(body)
                if text:
                    return text
        except Exception:
            continue
    return None
DOMAIN_SELECTORS = {'dantri.com.vn': ['div.singular-content', 'div.dt-news__content', 'article'], 'laodong.vn': ['div.article-content', 'div.art-body', 'div.content-wrapper', 'article'], 'vov.vn': ['div.article-content', 'div.row.text-long', 'article'], 'vtv.vn': ['div.ta-content', 'div.noidung', 'article']}
GENERIC_SELECTORS = ["[itemprop='articleBody']", 'article', '.article-content', '.article-body', '.article__body', '.detail-content', '.detail__content', '.news-content', '.post-content', '.entry-content', '.content-detail']

def extract_dom(soup, domain):
    for tag in soup(['script', 'style', 'noscript', 'nav', 'footer']):
        tag.decompose()
    selectors = DOMAIN_SELECTORS.get(str(domain), []) + GENERIC_SELECTORS
    body = None
    for selector in selectors:
        try:
            body = soup.select_one(selector)
            if body:
                break
        except Exception:
            continue
    if not body:
        return None
    paragraphs = []
    for p in body.find_all(['p', 'h2', 'h3'], recursive=True):
        text = re.sub('\\s+', ' ', p.get_text(' ', strip=True)).strip()
        if len(text) >= 30:
            paragraphs.append(text)
    return clean_text('\n\n'.join(paragraphs))

def extract_html(html, domain):
    if not html:
        return None
    try:
        soup = BeautifulSoup(html, 'html.parser')
        text = extract_json_ld(soup)
        if text:
            return text
        return extract_dom(soup, domain)
    except Exception:
        return None

def scrape_requests(url, domain):
    try:
        response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        if response.status_code != 200:
            return None
        return extract_html(response.content, domain)
    except Exception:
        return None

def run_requests_pass():
    targets = missing_indices()
    if not targets:
        return 0
    recovered = 0
    with ThreadPoolExecutor(max_workers=REQUEST_WORKERS) as executor:
        future_map = {executor.submit(scrape_requests, df.at[idx, 'real_url'], df.at[idx, 'outlet_domain']): idx for idx in targets}
        for future in tqdm(as_completed(future_map), total=len(future_map), desc='Requests'):
            idx = future_map[future]
            try:
                text = future.result()
                if update_row(idx, text, 'requests'):
                    recovered += 1
            except Exception:
                pass
    return recovered

def scrape_cloudscraper(url, domain):
    try:
        scraper = cloudscraper.create_scraper()
        response = scraper.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        if response.status_code != 200:
            return None
        return extract_html(response.content, domain)
    except Exception:
        return None

def run_cloudscraper_pass():
    targets = missing_indices()
    if not targets:
        return 0
    recovered = 0
    for idx in tqdm(targets, desc='Cloudscraper'):
        text = scrape_cloudscraper(df.at[idx, 'real_url'], df.at[idx, 'outlet_domain'])
        if update_row(idx, text, 'cloudscraper'):
            recovered += 1
        time.sleep(random.uniform(0.2, 0.5))
    return recovered

def create_driver():
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--disable-extensions')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument(f"user-agent={HEADERS['User-Agent']}")
    chrome_path = shutil.which('google-chrome') or shutil.which('chromium') or shutil.which('chromium-browser')
    if chrome_path:
        options.binary_location = chrome_path
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def run_selenium():
    targets = missing_indices()
    if not targets:
        return 0
    print(f'Selenium targets: {len(targets):,}')
    driver = create_driver()
    recovered = 0
    try:
        for idx in tqdm(targets, desc='Selenium'):
            try:
                driver.get(df.at[idx, 'real_url'])
                time.sleep(2.0)
                text = extract_html(driver.page_source, df.at[idx, 'outlet_domain'])
                if update_row(idx, text, 'selenium'):
                    recovered += 1
            except Exception:
                pass
            if recovered % 10 == 0 and recovered > 0:
                save_progress()
    finally:
        driver.quit()
    return recovered
print(f'Tổng số bài: {len(df):,}')
print(f'Ban đầu thiếu: {len(missing_indices()):,}')
for attempt in range(1, SIMPLE_PASSES + 1):
    before = len(missing_indices())
    if before == 0:
        break
    recovered = run_newspaper_pass()
    save_progress()
    after = len(missing_indices())
    print(f'Newspaper pass {attempt}: +{recovered:,} | còn {after:,}')
    if recovered == 0:
        break
for attempt in range(1, SIMPLE_PASSES + 1):
    before = len(missing_indices())
    if before == 0:
        break
    recovered = run_requests_pass()
    save_progress()
    after = len(missing_indices())
    print(f'Requests pass {attempt}: +{recovered:,} | còn {after:,}')
    if recovered == 0:
        break
for attempt in range(1, SIMPLE_PASSES + 1):
    if not missing_indices():
        break
    recovered = run_cloudscraper_pass()
    save_progress()
    remaining = len(missing_indices())
    print(f'Cloudscraper pass {attempt}: +{recovered:,} | còn {remaining:,}')
    if recovered == 0:
        break
if missing_indices():
    recovered = run_selenium()
    save_progress()
    print(f'Selenium: +{recovered:,}')
df['char_count'] = df['full_text'].fillna('').astype(str).str.len()
save_progress()
missing_final = (df['char_count'] < MIN_TEXT_CHARS).sum()
complete = len(df) - missing_final
print()
print(f'Hoàn tất: {complete:,} / {len(df):,} bài')
print(f'Missing: {missing_final:,} ({missing_final / len(df) * 100:.2f}%)')
print()
print(df['scrape_method'].replace('', 'missing').value_counts())
files.download(OUTPUT_FILE)
